# ARTI 402 — Deep Learning
## Lab 2 — Submission (Activations, Loss, and How a Network Learns)
**Exercises (code) and Assessment**

In [ ]:

import sys
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
np.set_printoptions(precision=4, suppress=True)

def vertical_data(samples, classes):
    X = np.zeros((samples * classes, 2))
    y = np.zeros(samples * classes, dtype="uint8")
    for class_number in range(classes):
        ix = range(samples * class_number, samples * (class_number + 1))
        X[ix] = np.c_[np.random.randn(samples) * 0.1 + class_number / 3,
                      np.random.randn(samples) * 0.1 + 0.5]
        y[ix] = class_number
    return X, y

In [2]:
### Exercise 1 — collapse the network
X = np.array([[ 1.0,  2.0,  3.0,  2.5],
              [ 2.0,  5.0, -1.0,  2.0],
              [-1.5,  2.7,  3.3, -0.8]])
rng = np.random.default_rng(402)
W1 = rng.normal(0, 0.5, size=(5, 4))
b1 = np.zeros(5)
W2 = rng.normal(0, 0.5, size=(3, 5))
b2 = np.zeros(3)
layer1_out = np.dot(X, W1.T) + b1
two_layer  = np.dot(layer1_out, W2.T) + b2

W_eq = W2 @ W1
one_layer = np.dot(X, W_eq.T) + b2

In [3]:
### Exercise 2 — implement ReLU and sigmoid
def relu(x):
    return np.maximum(0, x)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [4]:
### Exercise 3 — build Layer_Dense
class Layer_Dense:
    def __init__(self, n_inputs, n_neurons):
        self.weights = 0.1 * np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))

    def forward(self, inputs):
        self.output = np.dot(inputs, self.weights) + self.biases
        return self.output

In [5]:
### Exercise 4 — implement softmax
def softmax(x):
    shifted = x - np.max(x, axis=1, keepdims=True)
    exp_values = np.exp(shifted)
    probabilities = exp_values / np.sum(exp_values, axis=1, keepdims=True)
    return probabilities

In [6]:
### Exercise 5 — implement categorical cross-entropy
def categorical_crossentropy(y_pred, y_true):
    y_pred_clipped = np.clip(y_pred, 1e-7, 1 - 1e-7)
    correct_confidences = y_pred_clipped[range(len(y_pred)), y_true]
    return np.mean(-np.log(correct_confidences))

In [7]:
### Exercise 6 — measure a slope
def numerical_derivative(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

In [8]:
### Exercise 7 — descend the curve
f = lambda x: x**2 - 4*x + 5
x = 8.0
learning_rate = 0.1
path = [x]

for step in range(50):
    slope = numerical_derivative(f, x)
    x = x - learning_rate * slope
    path.append(x)

In [9]:
### Exercise 8 — backward pass through a neuron
x_val = [ 1.0, -2.0, 3.0]
w_val = [-3.0, -1.0, 2.0]
b_val = 1.0

z = x_val[0]*w_val[0] + x_val[1]*w_val[1] + x_val[2]*w_val[2] + b_val
y = max(z, 0)

dvalue = 1.0
drelu = dvalue if z > 0 else 0.0
dw = [x_val[0] * drelu, x_val[1] * drelu, x_val[2] * drelu]
dx = [w_val[0] * drelu, w_val[1] * drelu, w_val[2] * drelu]
db = 1.0 * drelu

## Assessment

In [10]:
### Assessment Setup
np.random.seed(0)
X_train, y_train = vertical_data(samples=100, classes=3)

def numerical_gradient(param, loss_fn, h=1e-4):
    grad = np.zeros_like(param)
    it = np.nditer(param, flags=["multi_index"])
    while not it.finished:
        idx = it.multi_index
        original = param[idx]
        param[idx] = original + h
        loss_plus = loss_fn()
        param[idx] = original - h
        loss_minus = loss_fn()
        param[idx] = original
        grad[idx] = (loss_plus - loss_minus) / (2 * h)
        it.iternext()
    return grad

In [11]:
### Q1 — Build the network
np.random.seed(42)
dense1 = Layer_Dense(2, 8)
dense2 = Layer_Dense(8, 3)

def forward(inputs):
    layer1_out = dense1.forward(inputs)
    layer1_activation = relu(layer1_out)
    layer2_out = dense2.forward(layer1_activation)
    probabilities = softmax(layer2_out)
    return probabilities

In [12]:
### Q2 — Loss and accuracy before training
start_probs = forward(X_train)
start_loss = categorical_crossentropy(start_probs, y_train)
start_acc = np.mean(np.argmax(start_probs, axis=1) == y_train)

In [13]:
### Q3 — Train it
def loss_fn():
    return categorical_crossentropy(forward(X_train), y_train)

params = [dense1.weights, dense1.biases, dense2.weights, dense2.biases]
learning_rate = 1.0
epochs = 150

loss_history = []
acc_history = []

for epoch in range(epochs):
    current_probs = forward(X_train)
    loss_history.append(loss_fn())
    acc_history.append(np.mean(np.argmax(current_probs, axis=1) == y_train))
    
    grads = [numerical_gradient(p, loss_fn) for p in params]
    
    for p, g in zip(params, grads):
        p -= learning_rate * g

    if epoch % 30 == 0:
        print(f"epoch {epoch:3d}   loss {loss_history[-1]:.4f}   acc {acc_history[-1]:.4f}")

epoch   0   loss 1.0971   acc 0.4700
epoch  30   loss 0.5933   acc 0.8800
epoch  60   loss 0.3788   acc 0.9033
epoch  90   loss 0.2692   acc 0.9100
epoch 120   loss 0.1947   acc 0.9200
